In [ ]:
import time
from datetime import datetime, timedelta
import os
import numpy as np
import pandas as pd
import requests


from web3 import Web3

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
df = pd.read_csv('data/eth_blocks_2025-01-01_2026-02-15.csv')
df

,timestamp,height,hash,size,gas_limit,gas_used,transaction_count,base_fee_per_gas
0,2026-02-15 00:32:35 UTC,24458637,0x173fa6c45072b36b3b2489bf945c813f5eeb2fb7b7ce...,123436,59999886,17391909,108,35527943
1,2026-02-15 16:49:47 UTC,24463507,0xe0c7adb0bd468c81f23eb23627e60e5e373f1e1850da...,136110,59941351,24197479,272,46609559
2,2026-02-15 05:17:23 UTC,24460056,0xaf0bfcf9fecc2e7999c498184bbb9a36c50fcddd59a4...,206574,59941237,52959002,166,46207812
3,2026-02-15 14:31:35 UTC,24462819,0x8e9a8703ee7c65965eb2c4717ca74f9c48442f5d2542...,151227,59999886,42068893,263,33619778
4,2026-02-15 07:24:23 UTC,24460690,0xdca86811dbd9c17587a3b01bf72e16f1d633bc9d6ccb...,170208,59999886,46730062,481,48904977
...,...,...,...,...,...,...,...,...
2939758,2026-01-16 00:49:11 UTC,24243802,0xe03f9bf772619c82fc0a37bd64d25bbece1bbc3b9033...,236213,60000000,58358979,1156,40700653
2939759,2026-01-16 23:17:47 UTC,24250526,0xc3d889f810a1730f34473aaf4e6d463295d9180da2ee...,575739,60000000,59985849,178,29792259
2939760,2026-01-16 20:21:23 UTC,24249648,0x090cb4a592df450e965f775cbefd085620312cc003c4...,63704,60000000,12596083,154,50460437
2939761,2026-01-16 00:19:35 UTC,24243655,0xfc34e3be6633a3849bcc5e6155cf21ead59165682509...,83523,60000000,20000456,159,48073799


In [4]:
df.describe()

,height,size,gas_limit,gas_used,transaction_count,base_fee_per_gas
count,2.939763e+06,2.939763e+06,2.939763e+06,2.939763e+06,2.939763e+06,2.939763e+06
mean,2.299577e+07,1.090783e+05,4.308254e+07,2.181120e+07,2.142717e+02,1.827983e+09
std,8.486366e+05,6.468409e+04,9.629040e+06,1.120064e+07,1.100360e+02,6.814647e+09
min,2.152589e+07,1.163000e+03,2.997070e+07,0.000000e+00,0.000000e+00,8.703865e+06
25%,2.226083e+07,7.005600e+04,3.596484e+07,1.432256e+07,1.490000e+02,1.561048e+08
50%,2.299577e+07,9.753000e+04,4.486800e+07,1.974043e+07,1.960000e+02,4.594324e+08
75%,2.373071e+07,1.336985e+05,4.504390e+07,2.730356e+07,2.560000e+02,1.250494e+09
max,2.446565e+07,1.956743e+06,6.011724e+07,6.005850e+07,2.753000e+03,8.255167e+11


In [6]:
print("=" * 60)
print("SHAPE")
print("=" * 60)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

print("\n" + "=" * 60)
print("COLUMNS & DTYPES")
print("=" * 60)
print(df.dtypes)

print("\n" + "=" * 60)
print("MEMORY USAGE")
print("=" * 60)
print(df.memory_usage(deep=True).to_string())
print(f"\nTotal: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct": missing_pct})
print(missing_df[missing_df["count"] > 0] if missing_df["count"].sum() > 0 else "No missing values")

print("\n" + "=" * 60)
print("DUPLICATES")
print("=" * 60)
print(f"Duplicate rows: {df.duplicated().sum():,}")


print("\n" + "=" * 60)
print("UNIQUE VALUES PER COLUMN")
print("=" * 60)
for col in df.columns:
    n = df[col].nunique()
    print(f"  {col}: {n:,} unique")

SHAPE
Rows: 2,939,763  |  Columns: 8

COLUMNS & DTYPES
timestamp              str
height               int64
hash                   str
size                 int64
gas_limit            int64
gas_used             int64
transaction_count    int64
base_fee_per_gas     int64
dtype: object

MEMORY USAGE
Index                      132
timestamp            235181040
height                23518104
hash                 361590849
size                  23518104
gas_limit             23518104
gas_used              23518104
transaction_count     23518104
base_fee_per_gas      23518104

Total: 703.70 MB

MISSING VALUES
No missing values

DUPLICATES
Duplicate rows: 0

UNIQUE VALUES PER COLUMN
  timestamp: 2,939,763 unique
  height: 2,939,763 unique
  hash: 2,939,763 unique
  size: 304,168 unique
  gas_limit: 83,549 unique
  gas_used: 2,794,940 unique
  transaction_count: 1,853 unique
  base_fee_per_gas: 2,935,682 unique


In [23]:
w3 = Web3(Web3.HTTPProvider(os.getenv("EL_NODE_URL")))

In [24]:
block = w3.eth.get_block("latest")
print(block.number)

24593524


In [ ]:
API_KEY = os.getenv("COINGECKO_API_KEY")

url = "https://api.coingecko.com/api/v3/simple/price"

params = {
    "ids": "ethereum",
    "vs_currencies": "usd"
}

headers = {
    "x-cg-demo-api-key": API_KEY
}

response = requests.get(url, params=params, headers=headers)

data = response.json()

eth_price = data["ethereum"]["usd"]

print(f"Current ETH price: ${eth_price}")

Current ETH price: $2075.69


In [ ]:
API_KEY = os.getenv("COINGECKO_API_KEY")

BASE_URL = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart/range"

headers = {
    "x-cg-demo-api-key": API_KEY
}

start_date = datetime(2025, 3, 6)
end_date = datetime(2026, 3, 5)

chunk_days = 90

rows = []

current_start = start_date

while current_start < end_date:

    current_end = min(current_start + timedelta(days=chunk_days), end_date)
    print(f"Fetching data from {current_start.date()} to {current_end.date()}...")
    
    params = {
        "vs_currency": "usd",
        "from": int(current_start.timestamp()),
        "to": int(current_end.timestamp())
    }

    response = requests.get(BASE_URL, params=params, headers=headers)
    data = response.json()

    prices = data["prices"]
    market_caps = data["market_caps"]
    volumes = data["total_volumes"]

    for i in range(len(prices)):
        timestamp = prices[i][0]

        rows.append({
            "timestamp": timestamp,
            "datetime": pd.to_datetime(timestamp, unit="ms"),
            "price": prices[i][1],
            "market_cap": market_caps[i][1],
            "volume": volumes[i][1]
        })

    print(f"Loaded {current_start.date()} → {current_end.date()}")

    current_start = current_end
    time.sleep(1)

df = pd.DataFrame(rows)

df = df.sort_values("datetime").reset_index(drop=True)

print(df.head())

Fetching data from 2025-03-06 to 2025-06-04...
Loaded 2025-03-06 → 2025-06-04
Fetching data from 2025-06-04 to 2025-09-02...
Loaded 2025-06-04 → 2025-09-02
Fetching data from 2025-09-02 to 2025-12-01...
Loaded 2025-09-02 → 2025-12-01
Fetching data from 2025-12-01 to 2026-03-01...
Loaded 2025-12-01 → 2026-03-01
Fetching data from 2026-03-01 to 2026-03-05...
Loaded 2026-03-01 → 2026-03-05
       timestamp                datetime        price    market_cap  \
0  1741208564423 2025-03-05 21:02:44.423  2230.584518  2.693445e+11   
1  1741212295383 2025-03-05 22:04:55.383  2236.766636  2.697674e+11   
2  1741215784773 2025-03-05 23:03:04.773  2239.435736  2.699957e+11   
3  1741219748187 2025-03-06 00:09:08.187  2244.935432  2.706242e+11   
4  1741223086191 2025-03-06 01:04:46.191  2239.385254  2.699733e+11   

         volume  
0  2.335307e+10  
1  2.253690e+10  
2  2.218404e+10  
3  2.222023e+10  
4  2.234494e+10  


In [17]:
df

,timestamp,datetime,price,market_cap,volume
0,1741208564423,2025-03-05 21:02:44.423,2230.584518,2.693445e+11,2.335307e+10
1,1741212295383,2025-03-05 22:04:55.383,2236.766636,2.697674e+11,2.253690e+10
2,1741215784773,2025-03-05 23:03:04.773,2239.435736,2.699957e+11,2.218404e+10
3,1741219748187,2025-03-06 00:09:08.187,2244.935432,2.706242e+11,2.222023e+10
4,1741223086191,2025-03-06 01:04:46.191,2239.385254,2.699733e+11,2.234494e+10
...,...,...,...,...,...
8731,1772640207826,2026-03-04 16:03:27.826,2150.392505,2.588748e+11,3.143937e+10
8732,1772643799382,2026-03-04 17:03:19.382,2143.077149,2.586854e+11,3.120427e+10
8733,1772647410540,2026-03-04 18:03:30.540,2147.037185,2.592250e+11,3.166291e+10
8734,1772650975057,2026-03-04 19:02:55.057,2162.208620,2.609615e+11,3.140737e+10


In [45]:
data

{'Data': [{'UNIT': 'HOUR',
   'TIMESTAMP': 1733950800,
   'OPEN': 3838.0773801909,
   'HIGH': 3838.5335069339,
   'LOW': 3817.32521945626,
   'CLOSE': 3833.13032121236,
   'VOLUME': 0,
   'QUOTE_VOLUME': 0},
  {'UNIT': 'HOUR',
   'TIMESTAMP': 1733954400,
   'OPEN': 3833.13032121236,
   'HIGH': 3848.14573277353,
   'LOW': 3826.47327068305,
   'CLOSE': 3839.44311558106,
   'VOLUME': 0,
   'QUOTE_VOLUME': 0},
  {'UNIT': 'HOUR',
   'TIMESTAMP': 1733958000,
   'OPEN': 3839.44311558106,
   'HIGH': 3851.41476879634,
   'LOW': 3828.22753640354,
   'CLOSE': 3834.39842449362,
   'VOLUME': 0,
   'QUOTE_VOLUME': 0},
  {'UNIT': 'HOUR',
   'TIMESTAMP': 1733961600,
   'OPEN': 3834.39842449362,
   'HIGH': 3839.83730811058,
   'LOW': 3810.68852970331,
   'CLOSE': 3820.52981580043,
   'VOLUME': 0,
   'QUOTE_VOLUME': 0},
  {'UNIT': 'HOUR',
   'TIMESTAMP': 1733965200,
   'OPEN': 3820.52981580043,
   'HIGH': 3826.42815633628,
   'LOW': 3800.68857808515,
   'CLOSE': 3807.94699811248,
   'VOLUME': 0,
   'QUO

In [83]:
BASE_URL = "https://data-api.coindesk.com/index/cc/v1/historical/minutes"

headers = {
    "Authorization": f"Apikey {API_KEY}"
}

market = "cadli"
instrument = "ETH-USD"

start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 1, 26, 21, 27, 0)

to_ts = int(end_date.timestamp())

limit = 2000

# rows = []

while True:

    params = {
        "market": market,
        "instrument": instrument,
        "limit": limit,
        "to_ts": to_ts,
        "groups": "OHLC,VOLUME",
        "aggregate": 1
    }

    r = requests.get(BASE_URL, params=params, headers=headers)
    data = r.json()

    batch = data["Data"]

    if not batch:
        break

    rows.extend(batch)

    oldest_ts = batch[0]["TIMESTAMP"]

    print("Downloaded until:", datetime.fromtimestamp(oldest_ts))

    if oldest_ts <= int(start_date.timestamp()):
        break
    to_ts = oldest_ts - 60

    time.sleep(1)


df = pd.DataFrame(rows)
df["datetime"] = pd.to_datetime(df["TIMESTAMP"], unit="s")
df = df.sort_values("datetime")

df = df[
    (df["datetime"] >= start_date) &
    (df["datetime"] <= end_date)
]

df = df.rename(columns={
    "OPEN": "open",
    "HIGH": "high",
    "LOW": "low",
    "CLOSE": "close",
    "VOLUME": "volume"
})

df = df.drop_duplicates(subset=["TIMESTAMP"])

print(df.shape)
print(df.head())
print(df.tail())

df.to_csv("eth_minutes_2025_2026.csv", index=False)

Downloaded until: 2025-01-25 12:08:00
Downloaded until: 2025-01-24 02:48:00
Downloaded until: 2025-01-22 17:28:00
Downloaded until: 2025-01-21 08:08:00
Downloaded until: 2025-01-19 22:48:00
Downloaded until: 2025-01-18 13:28:00
Downloaded until: 2025-01-17 04:08:00
Downloaded until: 2025-01-15 18:48:00
Downloaded until: 2025-01-14 09:28:00
Downloaded until: 2025-01-13 00:08:00
Downloaded until: 2025-01-11 14:48:00
Downloaded until: 2025-01-10 05:28:00
Downloaded until: 2025-01-08 20:08:00
Downloaded until: 2025-01-07 10:48:00
Downloaded until: 2025-01-06 01:28:00
Downloaded until: 2025-01-04 16:08:00
Downloaded until: 2025-01-03 06:48:00
Downloaded until: 2025-01-01 21:28:00
Downloaded until: 2024-12-31 12:08:00
(37288, 15)
          UNIT   TIMESTAMP         open         high          low  \
592892  MINUTE  1735689600  3332.507632  3332.575058  3331.424175   
592893  MINUTE  1735689660  3332.575058  3333.252526  3332.369019   
592894  MINUTE  1735689720  3333.237763  3336.691150  3333.

In [87]:
df = pd.DataFrame(rows)
df["datetime"] = pd.to_datetime(df["TIMESTAMP"], unit="s")
df = df.sort_values("datetime")

df = df.rename(columns={
    "OPEN": "open",
    "HIGH": "high",
    "LOW": "low",
    "CLOSE": "close",
    "VOLUME": "volume"
})

df = df.drop_duplicates(subset=["TIMESTAMP"])
df

,UNIT,TIMESTAMP,open,high,low,close,volume,QUOTE_VOLUME,VOLUME_TOP_TIER,QUOTE_VOLUME_TOP_TIER,VOLUME_DIRECT,QUOTE_VOLUME_DIRECT,VOLUME_TOP_TIER_DIRECT,QUOTE_VOLUME_TOP_TIER_DIRECT,datetime
592000,MINUTE,1735636080,3378.569323,3380.485797,3378.217773,3380.465130,1767.621519,5.974709e+06,969.995245,3.278723e+06,88.282421,298298.946808,87.683491,296273.130839,2024-12-31 09:08:00
592001,MINUTE,1735636140,3380.465130,3382.087644,3379.523548,3381.908966,2500.374313,8.453734e+06,1622.858775,5.487008e+06,133.230708,450299.577347,132.825178,448927.966293,2024-12-31 09:09:00
592002,MINUTE,1735636200,3381.908966,3383.238625,3381.428658,3382.847440,2221.012152,7.513716e+06,1247.153641,4.219135e+06,188.263742,636652.513877,184.458903,623698.905942,2024-12-31 09:10:00
592003,MINUTE,1735636260,3382.847440,3383.516298,3382.149028,3382.240800,1515.060300,5.125635e+06,867.835997,2.936108e+06,85.229725,288205.770504,85.036495,287551.856548,2024-12-31 09:11:00
592004,MINUTE,1735636320,3382.240800,3383.575566,3381.438189,3381.591191,1391.147431,4.706566e+06,819.917387,2.773771e+06,78.714229,266118.741873,78.559561,265595.298606,2024-12-31 09:12:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,MINUTE,1771275360,1993.592348,1993.592436,1991.998667,1992.059820,2449.901777,4.879452e+06,1042.716522,2.076631e+06,169.015468,336451.804526,162.860668,324202.884306,2026-02-16 20:56:00
1996,MINUTE,1771275420,1992.059820,1993.146110,1991.808684,1992.881738,2614.963938,5.208310e+06,1282.120863,2.553640e+06,188.397666,375039.166557,187.591856,373435.571706,2026-02-16 20:57:00
1997,MINUTE,1771275480,1992.881738,1993.668483,1992.616035,1993.666354,2378.978058,4.741151e+06,1151.227796,2.294991e+06,157.786077,314198.056788,157.553098,313734.635499,2026-02-16 20:58:00
1998,MINUTE,1771275540,1993.666354,1994.365204,1993.646374,1993.967930,3014.687517,6.009953e+06,1534.545144,3.059086e+06,222.362850,443097.765103,218.906048,436208.518704,2026-02-16 20:59:00


In [88]:
df['datetime'].min(), df['datetime'].max()

(Timestamp('2024-12-31 09:08:00'), Timestamp('2026-02-16 21:00:00'))

In [89]:
df.to_csv("data/eth_minutes_2025_2026.csv", index=False)

In [ ]:
df = df.sort_values("datetime").reset_index(drop=True)

# вычисляем разницу между соседними часами
df["diff_minutes"] = df["datetime"].diff().dt.total_seconds() / 60

# проверяем, есть ли пропуски
missing_hours = df[df["diff_minutes"] != 1]

if missing_hours.empty:
    print("Все часы на месте! Пропусков нет.")
else:
    print("Есть пропуски часов:")
    print(missing_hours[["datetime", "diff_minutes"]])

Есть пропуски часов:
             datetime  diff_minutes
0 2024-12-31 09:08:00           NaN
